# RAG CoT
(Chain of Thought)

RAG 파이프라인에서 LLM이 단순한 정보 조합을 넘어서 단계적 사고를 통해 논리적 답변을 할 수 있도록 한다.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

In [2]:
# 가상 검색기
from langchain_core.documents import Document

def retrieve_vectordb(query=None):
    return [
        Document(page_content='대한민국의 수도는 서울입니다. 서울은 한강을 끼고 발달한 도시입니다.'),
        Document(page_content='서울의 대표적인 관광지는 경복궁, 남산타워, 명동 등이 있습니다.'),
        Document(page_content='서울의 인구는 약 천만명이고, 교통 문화 인프라가 잘 갖추어져 있습니다.'),
    ]

retrieve_vectordb()


[Document(metadata={}, page_content='대한민국의 수도는 서울입니다. 서울은 한강을 끼고 발달한 도시입니다.'),
 Document(metadata={}, page_content='서울의 대표적인 관광지는 경복궁, 남산타워, 명동 등이 있습니다.'),
 Document(metadata={}, page_content='서울의 인구는 약 천만명이고, 교통 문화 인프라가 잘 갖추어져 있습니다.')]

In [6]:
# 채팅 프롬프트 / 휴먼 메시지 템플릿
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser # 출력 -> 문자열 파싱

# 함수를 Runnable로 감싸 chain에서 실행
from langchain_core.runnables import RunnableLambda

llm = init_chat_model('gpt-5.6-luna')

prompt = ChatPromptTemplate.from_template(''' 
당신은 데이터를 분석해서 논리적인 결론을 도출하는 전문가 챗봇입니다.
아래 [검색된 문서]를 바탕으로 사용자의 [질문]에 대해 답변하세요.

[검색된 문서]
{context}

[질문]
{question}

[지시사항]
다음의 단계에 따라 사고한 후, 답변을 작성하세요.
1. **핵심데이터 정리**: 문서에서 사용자 질문과 관련한 팩트를 추출해보세요.
2. **상호관계 분석**: 각 항목별로 어떤 상관관계/시너지를 도출하는지 고민하세요.
3. **논리적 서술**: 위의 사고한 내용을 토대로 사용자 질문에 대한 답변을 준비하세요.
4. **최종 답변**: 서론-본론-결론 구조에 맞춰 완성된 답변을 작성하세요.
''')

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

question = '서울의 인구, 관광지, 교통인프라를 종합해서 여행하기 좋은 이유를 논리적으로 설명해주세요.'
retrieved_docs = retrieve_vectordb(question)
# 문서 본문만 뽑아서 하나의 문자열 context로 생성
context = '\n\n'.join( [doc.page_content for doc in retrieved_docs])

response = chain.invoke({'context': context, 'question':question})
print(response)

### 서론  
서울은 약 천만 명이 거주하는 대도시로, 관광지와 교통·문화 인프라가 잘 갖추어져 있어 여행하기 좋은 도시입니다. 특히 역사, 전망, 쇼핑 등 서로 다른 매력을 가진 관광지를 편리하게 방문할 수 있다는 점이 큰 장점입니다.

### 본론  

1. **핵심 데이터 정리**  
   - 서울의 인구는 약 천만 명입니다.  
   - 대표적인 관광지로 경복궁, 남산타워, 명동 등이 있습니다.  
   - 교통과 문화 인프라가 잘 갖추어져 있습니다.  
   - 서울은 한강을 중심으로 발달한 도시입니다.  

2. **항목 간 상호관계**  
   - 많은 인구를 기반으로 다양한 교통·문화 시설이 발달했을 가능성이 높으며, 실제로 서울은 교통과 문화 인프라가 잘 갖추어져 있습니다.  
   - 경복궁에서는 역사와 전통문화를, 남산타워에서는 서울의 도시 경관을, 명동에서는 쇼핑과 도심 문화를 경험할 수 있어 관광 선택의 폭이 넓습니다.  
   - 잘 갖추어진 교통 인프라는 이러한 관광지들을 효율적으로 이동할 수 있게 해 여행의 편의성을 높입니다.  
   - 또한 한강을 끼고 발달한 도시라는 점은 관광지 방문과 함께 서울의 도시 경관과 여가 공간을 즐길 수 있는 배경이 됩니다.  

### 결론  
서울은 약 천만 명이 거주하는 대도시답게 교통과 문화 인프라가 잘 마련되어 있으며, 경복궁·남산타워·명동처럼 역사, 전망, 쇼핑을 아우르는 다양한 관광지가 있습니다. 따라서 여행자는 편리하게 이동하면서도 여러 형태의 관광을 한 도시 안에서 경험할 수 있습니다. 이러한 관광지의 다양성과 편리한 인프라가 서울을 여행하기 좋은 도시로 만드는 핵심 이유입니다.
